# Einfach Deutsch — Colab Training Notebook

This notebook trains the German text-simplification models end-to-end on Google Colab.

Recommended runtime: **GPU T4** (Change runtime type → Hardware accelerator → GPU).

The notebook will:
1. Mount Google Drive.
2. Clone or update the repository.
3. Install dependencies.
4. Auto-fetch the [Klexikon](https://huggingface.co/datasets/dennlinger/klexikon) German simplification dataset.
5. Train the mT5-small baseline.
6. Train the Mistral-7B LoRA adapter.

All checkpoints are saved under `/content/drive/MyDrive/einfach-deutsch/checkpoints/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update repository

Replace `YOUR_USERNAME` with your GitHub username. If the repo already exists, it is pulled instead.

In [ ]:
%cd /content
import os
repo_url = 'https://github.com/YOUR_USERNAME/einfach-deutsch.git'
if os.path.isdir('/content/einfach-deutsch'):
    %cd /content/einfach-deutsch
    !git pull
else:
    !git clone {repo_url}
    %cd /content/einfach-deutsch

## Install dependencies

In [ ]:
!pip install -q -U -r requirements.txt

## Verify GPU

In [ ]:
!nvidia-smi

## Prepare dataset

This downloads the Klexikon dataset from Hugging Face, builds aligned complex/simple sentence pairs, and writes the processed splits to `data/processed`.

In [ ]:
!python prepare_data.py --sources klexikon

## Train baseline mT5-small

Faster option; train this first to validate the pipeline.

In [ ]:
!python train_baseline.py --output-dir /content/drive/MyDrive/einfach-deutsch/checkpoints/baseline

## Train LoRA 7B model

On a free Colab T4 this uses 4-bit quantization. Expect a few hours for the full Klexikon run.

In [ ]:
!python train_lora.py --output-dir /content/drive/MyDrive/einfach-deutsch/checkpoints/lora

## Evaluate

Create `data/evaluation/test.csv` with `complex`, `simple`, and optionally `source` columns, then run evaluation.

In [ ]:
!python evaluate.py --test-file data/evaluation/dummy_test.csv --output-dir /content/drive/MyDrive/einfach-deutsch/outputs/evaluation